### Hybrid Retriever - Combine Dense and Sparse Retriever

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document

In [19]:
#Step 1: Sample Doc
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

#Step 2: Dense Retriever (FAISS+HuggingFace)
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore=FAISS.from_documents(docs,embedding_model)
dense_retriever=dense_vectorstore.as_retriever()


In [20]:
##Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3

#Step 4: Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)


In [21]:
#Step 5: Query and get results
query="How can I build an application using LLMs?"
results=hybrid_retriever.invoke(query)

#Step 6: Print Results
for i,doc in enumerate(results):
    print(f"\n{doc.page_content}")


LangChain helps build LLM applications.

Langchain can be used to develop agentic ai application.

Langchain has many types of retrievers.

Pinecone is a vector database for semantic search.


### RAG Pipeline with Hybrid Retriever

In [22]:
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [29]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [39]:
#Step 5: Prompt Template
prompt=PromptTemplate.from_template("""
You are a helpful RAG assistant.

Use ONLY the information from the context. 
You may rephrase, summarize, and combine the sentences logically.
DO NOT add any new facts not present in the context.
                                    
Context: {context}

Question: {input}
""")

#Step 6: LLM
llm=init_chat_model("groq:llama-3.1-8b-instant",temperature=0.6)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002F0C5A607C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002F0C5A61040>, model_name='llama-3.1-8b-instant', temperature=0.6, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
## Create stuff document chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

#Create a full RAG chain
rag_chain=create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002F0C53456D0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002F0C5344E10>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nYou are a helpful RAG assistant.\n\nUse ONLY the information from the context. \nYou may rephrase, summarize, and combine the se

In [41]:
#Step 9: Ask a question

query={"input":"How can I build an app using LLMs?"}
response=rag_chain.invoke(query)

#Step 10:Output
print(f"Question: {query["input"]}\nAnswer:\n",response["answer"])

print("\nSource Docs:")
for i,doc in enumerate(response["context"]):
    print(f"Metadata:{doc.metadata}, Page Content:{doc.page_content}")

Question: How can I build an app using LLMs?
Answer:
 To build an app using LLMs, you can use Langchain to develop agentic AI applications. Langchain provides various tools to create these applications.

Source Docs:
Metadata:{}, Page Content:LangChain helps build LLM applications.
Metadata:{}, Page Content:Langchain can be used to develop agentic ai application.
Metadata:{}, Page Content:Langchain has many types of retrievers.
Metadata:{}, Page Content:Pinecone is a vector database for semantic search.
